# dispatch-back-fn-from-recipe — worked example 1: Dispatch the back function for a node with one parent

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dispatch-back-fn-from-recipe`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

Every non-leaf tensor in an autograd system carries a `recipe` that records the forward function used to produce it and a `parents` dict mapping argnum → parent tensor. To run the backward pass, the engine looks up `back_funcs[(recipe.func, argnum)]` to find the correct gradient function for each parent. The lookup table separates the registry of gradient rules from the traversal logic.

## Worked solution

Say we have a node produced by `neg(x)`. Its recipe stores `func=neg` and `parents={0: x}` (only one parent, at argnum 0).

**Step 1.** Iterate `node.recipe.parents.items()`: yields `(0, x)`.

**Step 2.** Form the lookup key: `(neg, 0)`. Look it up in `back_funcs`: returns `neg_back`.

**Step 3.** Append `(0, x, neg_back)` to results.

**Result.** `[(0, x, neg_back)]`. The caller will call `neg_back(out_grad, out, x)` to compute the gradient for `x`.

The key insight: argnum disambiguates when a function has multiple parents with different gradient rules (e.g. `div(a, b)` has different back_fn for argnum 0 and argnum 1).

In [ ]:
from dataclasses import dataclass
from typing import Callable, Any

@dataclass
class Recipe:
    func: Callable
    parents: dict  # {argnum: parent_tensor}
    args: tuple = ()
    kwargs: dict = None

class FakeTensor:
    def __init__(self, name, recipe=None):
        self.name = name
        self.recipe = recipe
    def __repr__(self):
        return f'FakeTensor({self.name})'

# --- forward functions ---
def neg(x): return -x
def neg_back(out_grad, out, x): return -out_grad

# --- Build a simple node: y = neg(x) ---
x = FakeTensor('x')  # leaf
y = FakeTensor('y', Recipe(func=neg, parents={0: x}))

# --- Back-funcs registry ---
back_funcs = {(neg, 0): neg_back}

# --- Dispatch function ---
def dispatch_back_fns(node, back_funcs):
    results = []
    for argnum, parent in node.recipe.parents.items():
        back_fn = back_funcs[(node.recipe.func, argnum)]
        results.append((argnum, parent, back_fn))
    return results

triples = dispatch_back_fns(y, back_funcs)
print('Dispatch result:', triples)
print('argnum=0, parent is x:', triples[0][0] == 0 and triples[0][1] is x)
print('back_fn is neg_back:', triples[0][2] is neg_back)